# 17 · RAG basics: embeddings, chunking and cosine top-k
Level L3 · Part 3 · core · ~25 min · builds on 07

## Goal
Split the FTC mail-order rule guide into chunks, embed them, find the top-3 chunks for a customer's question by
cosine similarity in a vector store, and answer with only those chunks in the prompt, citing the chunk numbers you used.

## Why it matters
The support desk needs the FTC guide to answer "can I cancel and get my money back?" Without retrieval, there are
two bad choices. Paste the whole guide into every prompt, and every question pays for every word. Or let the model
answer from memory, which may be out of date and may invent a rule. Here is the size of the first choice.
**Nothing is sent to a model in this cell.**

In [1]:
from pathlib import Path

doc = Path("../data/ftc_30_day_rule.txt").read_text(encoding="utf-8")  # the FTC guide, plain text (see data/SOURCES.md)
words = len(doc.split())
print(f"whole guide: {words} words, about {int(round(words * 1.3, -2))} tokens (roughly 1.3 tokens per English word, a common estimate; the real count depends on the model's tokenizer)")
print("sent again with every question if you paste it in")

whole guide: 5029 words, about 6500 tokens (roughly 1.3 tokens per English word, a common estimate; the real count depends on the model's tokenizer)
sent again with every question if you paste it in


Retrieval sends only the few chunks that match the question. The cost stays small as the document grows, and each
answer can point to the text it used.

| Without retrieval | With retrieval (RAG) |
|---|---|
| Paste the whole guide into every prompt | Send the 3 chunks that match the question |
| The model answers from memory, with no source | The answer cites chunk numbers you can open |
| A new edition means re-pasting by hand | Re-embed the changed chunks |

**What it makes easy:** an answer that cites the text it used, from a document much larger than the prompt.
**When you don't need it:** Anthropic notes that for a knowledge base under about 200,000 tokens, putting everything in
the prompt may be simpler than RAG. This guide is small, so this lesson shows the mechanism on a small example.

## The idea
An **embedding** is a list of numbers (here 768 of them) that a model computes from a piece of text. Texts with similar
meaning get lists that point in similar directions. **Cosine similarity** measures that direction: close to 1 means
the same meaning, close to 0 means unrelated. **Top-k** means: score every chunk against the question, sort, and keep
the k best. **Chunking** cuts the document into pieces, each about one idea, small enough to fit in a prompt.

Analogy: a library card catalogue. Each card (a chunk) summarises one part of the collection. The librarian compares
your question with every card and hands you the three best cards, not the whole library.

## Picture
```mermaid
flowchart LR
  D[FTC guide] --> C[Chunks: whole paragraphs, 120 words]
  C --> E1[Embed each chunk once] --> V[(Vector store)]
  Q[Customer question] --> E2[Embed the question] --> S[Cosine top-k in the store]
  V --> S
  S --> K[Top 3 chunks] --> P[Prompt: question + numbered excerpts] --> A[Answer with citations]
```

## Step by step
**Step 1 · peek at the data and chunk it (no model call).** The example is the FTC's *Business Guide to the mail, internet
or telephone order merchandise rule* (How to Comply and Q&A, text only). It is a US federal government work, from
https://search.ftc.gov (see `data/SOURCES.md`). A text splitter cuts it only at blank lines, so each chunk is whole
paragraphs, up to 120 words. Each bullet is its own paragraph, so no bullet is cut in half.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter  # the splitter from LangChain's text-splitters package

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"],  # cut only at blank lines: whole paragraphs
    chunk_size=120, chunk_overlap=0,
    length_function=lambda text: len(text.split()),  # measure in words, not characters
    keep_separator=False)
chunks = splitter.split_text(doc)
print(doc[:120].replace("\n", " / "), "...")  # a peek: the guide starts with its "How to Comply" section
print("chunks:", len(chunks), "· chunk 0 starts:", repr(chunks[0][:50]))
print("chunk 20 starts:", repr(chunks[20][:50]))

How to Comply With the Rule /  / The following information will help you comply with the Rule. /  / What You Should Know Before  ...
chunks: 56 · chunk 0 starts: 'How to Comply With the Rule\n\nThe following informa'
chunk 20 starts: '- the reason for the delay, and\n\n- a statement tha'


**Step 2 · the clients (the setup).** The key is read from `.env`, never typed here. While a recording replays, a
placeholder is used and no key is sent. The embedding model is served by the Gemini API, so it gets its own client
(the Vertex express endpoint does not serve this embedding model for a batch); the answer uses the same client as
lesson 07. This notebook makes **two** model calls in total: one embedding request for the three questions, and one
answer. The chunk vectors are not computed here; step 3 loads them.
Embeddings use the Gemini API endpoint, answers use Vertex. In your project one client may be enough; check which
your key works with.

In [3]:
import sys; sys.path.insert(0, "..")
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("17_rag-basics")

import re
from google import genai  # the official Google SDK; in your project: genai.Client(api_key=...)
from google.genai import errors, types
from pydantic import BaseModel, Field
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_exponential_jitter
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

NO_KEY = "replay"  # course only: while replaying, the recording answers and this placeholder is never sent
REPLAYING = llm_replay.is_replaying()
KEY = NO_KEY if REPLAYING else env("GOOGLE_CLOUD_API_KEY")
gemini = genai.Client(vertexai=True, api_key=KEY, http_options=types.HttpOptions(timeout=30_000))  # answers (lesson 07)
embedder = genai.Client(api_key=KEY, http_options=types.HttpOptions(timeout=30_000))  # embeddings: Gemini API endpoint
EMBED_MODEL = "gemini-embedding-2"  # 768 dimensions: the size the Gemini docs recommend for this model
GEN_MODEL = "gemini-3.1-flash-lite"
DIMS = 768
print("client ready · replaying:", REPLAYING)

LLM mode: replay (17_rag-basics.yaml)


client ready · replaying: True


**Step 3 · load the chunk vectors and index them (no model call).** Each chunk is a vector of 768 numbers. The vectors
were computed once, when the lesson was recorded, and saved in `data/ftc_chunks_embeddings.json` (lesson 18 uses the
same file). Each chunk was embedded with the document-side prefix `title: FTC 30-day rule guide | text: ` (the
`prefix` in that file), so the recap's "embed the question and the chunks the same way" means the same model and size,
with the prefix for documents and the query prefix for questions. The file was made with one call over `chunks`:
`embedder.models.embed_content(model=EMBED_MODEL, contents=[prefix + c for c in chunks], config=...)`. For your own
document, do the same with your chunks. The cell checks that the saved text matches the chunks you just made. Then it
puts the vectors in a vector store: Chroma, an embedded store that ranks by cosine similarity.

In [4]:
import json
import chromadb
from chromadb.config import Settings

saved = json.loads(Path("../data/ftc_chunks_embeddings.json").read_text(encoding="utf-8"))
assert [c["text"] for c in saved["chunks"]] == chunks, "the saved vectors were made from different chunks"
chunk_vecs = [c["vector"] for c in saved["chunks"]]
print("model:", saved["model"], "· chunk vectors:", len(chunk_vecs), "x", len(chunk_vecs[0]))

store = chromadb.EphemeralClient(settings=Settings(anonymized_telemetry=False)).create_collection(
    name="ftc_guide", embedding_function=None,  # we pass our own vectors, so the store calls no model
    configuration={"hnsw": {"space": "cosine"}})  # rank by cosine similarity
store.add(ids=[str(i) for i in range(len(chunks))], documents=chunks, embeddings=chunk_vecs)
print("indexed chunks:", store.count())

model: gemini-embedding-2 · chunk vectors: 56 x 768


indexed chunks: 56


**Step 4 · embed the questions (model call 1), then ask the store for the top 3.** Each question gets the query prefix,
`task: search result | query:`, so the model knows it is a question and not a document. Chroma returns a cosine
*distance*, which is 1 minus the similarity, so the cell converts it back. `Hit` is a pydantic model: each result has
a checked type, so a bad value fails here and not later in the prompt.

In [5]:
QUESTIONS = [
    "A customer ordered a jacket 20 days ago and the shop now says it will ship in 45 days. Can they cancel for a full refund?",
    "What must a delay notice say?",
    "Does the rule cover orders taken by phone?",
]
QUERY_PREFIX = "task: search result | query: "
contents = [types.Content(parts=[types.Part.from_text(text=QUERY_PREFIX + q)]) for q in QUESTIONS]
result = embedder.models.embed_content(model=EMBED_MODEL, contents=contents,  # model call 1: one request, 3 vectors
                                       config=types.EmbedContentConfig(output_dimensionality=DIMS))
q_vecs = [e.values for e in result.embeddings]


class Hit(BaseModel):
    chunk: int
    score: float = Field(description="cosine similarity: 1 means the same direction, 0 means unrelated")
    text: str


def top_k(query_vec: list[float], k: int = 3) -> list[Hit]:
    res = store.query(query_embeddings=[query_vec], n_results=k)
    return [Hit(chunk=int(i), score=round(1 - d, 4), text=t)  # distance = 1 - similarity
            for i, d, t in zip(res["ids"][0], res["distances"][0], res["documents"][0])]


for q, qv in zip(QUESTIONS, q_vecs):
    print("Q:", q)
    for h in top_k(qv):
        print(f"   chunk {h.chunk:2d}  score {h.score:.3f}  {h.text[:60]!r}")

Q: A customer ordered a jacket 20 days ago and the shop now says it will ship in 45 days. Can they cancel for a full refund?
   chunk 21  score 0.760  'You must cancel an order and provide a prompt refund when:\n\n'
   chunk  8  score 0.754  'When you learn that you cannot ship on time, you must decide'
   chunk 15  score 0.748  'If you don’t want to wait, you may cancel your order and rec'
Q: What must a delay notice say?
   chunk 10  score 0.780  'In seeking your customer’s consent to delay, the first delay'
   chunk 12  score 0.774  'We will be unable to ship the merchandise listed above until'
   chunk 14  score 0.757  'For example, suppose you have a reasonable basis for being a'
Q: Does the rule cover orders taken by phone?
   chunk 54  score 0.763  'A: Yes. The shipment representation you make in negotiating '
   chunk 53  score 0.757  'Qualifying 30-day or Other Shipment Representations\n\nQ: In s'
   chunk 34  score 0.753  'The Rule also does not cover services, such as mail o

Read the output. For the first question, the top three are the cancel and refund rules (chunks 21, 8 and 15). Their
scores are close together, so the ranking is a narrow call. The third question's top chunk is a question and answer on
telephone orders. All three questions print in full above, so you can check the third chunk of question 2 and the top
chunk of question 3 against the text. A score is only a ranking: 0.76 does not mean "76% correct". That is why step 5
reads the chunks and checks the citations.

**Step 5 · answer with the retrieved text (model call 2).** The prompt holds the question and the three chunks,
numbered by chunk id. The system instruction says: answer only from the excerpts, cite them, and say so when they
do not answer. Then check the citations in code: each cited number must be one of the chunks you sent.

In [6]:
Q1 = QUESTIONS[0]
hits = top_k(q_vecs[0])
excerpts = "\n\n".join(f"[{h.chunk}] {h.text}" for h in hits)
SYSTEM = ("You are the Brightside Outfitters support desk. Answer only from the numbered excerpts. Cite the excerpt "
          "numbers you used, like [3]. If the excerpts do not answer the question, say so and do not guess.")
TRANSIENT = {408, 429, 500, 502, 503, 504}


@retry(retry=retry_if_exception(lambda e: isinstance(e, errors.APIError) and e.code in TRANSIENT),
       stop=stop_after_attempt(4), wait=wait_exponential_jitter(initial=5, max=30, jitter=5), reraise=True)
def ask(prompt: str):
    """One answer call. A 429 (too many requests) or 5xx is retried with growing waits, as in lesson 02."""
    return gemini.models.generate_content(model=GEN_MODEL, contents=prompt,
                                          config=types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0))


answer = ask(f"Question: {Q1}\n\nExcerpts:\n{excerpts}")  # model call 2
cited = {int(n) for n in re.findall(r"\[(\d+)\]", answer.text)}
retrieved = {h.chunk for h in hits}
print("answer:", answer.text)
print("retrieved chunks:", sorted(retrieved), "· cited:", sorted(cited), "· cites only retrieved chunks:", cited <= retrieved)
llm_replay.stop()  # the last model call of this notebook; nothing below calls a model

answer: Yes, you may cancel your order and receive a prompt refund if you do not wish to wait for the delayed shipment [15]. You can do this by calling our toll-free customer service number at (800) 555-1234 [15]. Additionally, you have the right to cancel an order at any time before the merchandise is shipped [21].
retrieved chunks: [8, 15, 21] · cited: [15, 21] · cites only retrieved chunks: True


Read the answer before you trust it. The phone number, (800) 555-1234, is not Brightside's. It comes from chunk 15,
which is an example notice in the FTC guide, so the answer repeats a template as fact. A cited answer can still do
this, so read the chunk. The check above proves the numbers were sent, not that the claim matches the chunk. Open
chunk 15 and read it. Only some retrieved chunks are cited (15 and 21) because the model used those; chunk 8 was
retrieved but the answer did not need it, so a retrieved chunk is not automatically a cited one.

## Under the hood
Chroma did the top-k for you. Here is the same ranking written by hand, so you can see every step: a dot product, two
lengths, a division and a sort. Then numpy does the same in one matrix product. Use the hand version to understand
the store, not in a real service.

In [7]:
import math


def cosine_py(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))


def top_k_py(question_vec: list[float], rows: list[list[float]], k: int = 3) -> list[tuple[int, float]]:
    scored = [(i, cosine_py(question_vec, row)) for i, row in enumerate(rows)]
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored[:k]


hand = top_k_py(q_vecs[0], chunk_vecs)
print("hand-written:", [(i, round(s, 3)) for i, s in hand])
print("Chroma:      ", [(h.chunk, h.score) for h in hits])
print("same chunks, same order:", [i for i, _ in hand] == [h.chunk for h in hits])

import numpy as np  # the same cosine as one matrix product: how numeric libraries do it
m = np.array(chunk_vecs)
m = m / np.linalg.norm(m, axis=1, keepdims=True)  # every chunk vector to length 1
q = np.array(q_vecs[0]); scores = m @ (q / np.linalg.norm(q))  # one dot product per chunk
print("numpy:       ", [(int(i), round(float(scores[i]), 3)) for i in np.argsort(-scores)[:3]])

hand-written: [(21, 0.76), (8, 0.754), (15, 0.748)]
Chroma:       [(21, 0.7604), (8, 0.7543), (15, 0.7479)]
same chunks, same order: True
numpy:        [(21, 0.76), (8, 0.754), (15, 0.748)]


## At work
- Anthropic's contextual retrieval post measured top-20 retrieval failures: 5.7% with plain embeddings, 1.9% with
  contextual embeddings, contextual BM25 and a reranker together. Plain cosine top-k is the first step, not the whole
  system → lesson 18 (hybrid search and reranking).
- Teams tune the chunk size and k on their own labelled questions and measure how often the right chunk is in the top k
  (recall at k). They do not judge by reading a few answers → lesson 08 (first eval) and lesson 22 (eval depth).
- Retrieved text is input from outside, and a document can contain "ignore your instructions". Treat it like a user
  message → lesson 30 (injection and sandboxing).
- Go deeper: at scale, the store adds approximate indexes and metadata filters; lesson 18 builds on this store.

## Common mistakes
1. **Comparing vectors from different models or sizes.** A question embedded at 3072 dimensions cannot be compared with
   chunks embedded at 768. The store refuses it, and the error below says so. Embed everything with the same model and
   `output_dimensionality`.

In [8]:
store.query(query_embeddings=[[0.1] * 3072], n_results=3)  # 3072 numbers against the 768-number chunk vectors

InvalidArgumentError: Collection expecting embedding with dimension of 768, got 3072

The fix is to embed the question with the same `EMBED_MODEL` and `DIMS` as the chunks. Vectors from one model
are not comparable with vectors from another.

2. **Chunks that are too big or too small.** A chunk of 2,000 words mixes several topics, so its score is an average
   and the right sentence is buried. A chunk of one line loses the context that gives it meaning. Start near 100 to
   200 words, then measure on your own questions (lesson 22).

## Try it
Write `keep_relevant(hits, min_score)`. It returns the hits whose score is at least `min_score`. If none are close
enough, it returns an empty list, so the desk can say "the guide does not cover this" instead of guessing. No model
call is needed: use the `hits` you already have.

In [9]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [10]:
def keep_relevant(hits: list[Hit], min_score: float) -> list[Hit]:
    return [h for h in hits if h.score >= min_score]  # an empty list when nothing is close enough


print([h.chunk for h in keep_relevant(hits, 0.5)])
print(keep_relevant(hits, 0.99))

[21, 8, 15]
[]


## Recap
- An embedding turns text into a vector; cosine similarity scores how close two vectors point, and top-k keeps the best.
- Chunk by whole paragraphs, about 100 to 200 words each, and embed the question and the chunks the same way.
- Answer from the top-k excerpts only, cite their numbers, and check the citations in code.
- The scores rank chunks; they are not a probability that the answer is right. Measure recall on your own questions.

**Interview:** "How do you find the most relevant chunks for a question?" Strong answer: "I embed the chunks once and the
question with the same model, rank the chunks by cosine similarity, and keep the top k. Then I measure recall at k on
labelled questions before I tune the chunk size."